In [15]:
import polars as pl

In [16]:
df.head()

openrouter_id,date,openrouter_author,openrouter_model_name,context_length,huggingface_id,prompt_price,completion_price,image_price,request_price,web_search_price,internal_reasoning_price,image_output_price,discount_price,hidden,endpoint_is_hidden,endpoint_is_deranked,endpoint_is_disabled,endpoint_provider,input_cache_read_price,audio_price,input_audio_cache_price,input_cache_write_price,image_token_price,audio_output_price,line_items_price,companyDiscount_price,vendorDiscount_price,model_permaslug,variant,variant_permaslug,count,total_completion_tokens,total_prompt_tokens,total_native_tokens_reasoning,num_media_prompt,num_media_completion,num_audio_prompt,total_native_tokens_cached,total_tool_calls,requests_with_tool_call_errors,model,volume,net_revenue,total_tokens,openrouter_id_clean,huggingface_author,huggingface_model_name
str,"datetime[ms, UTC]",str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,str,str,str
"""01-ai/yi-1.5-34b-chat""",2024-11-04 00:00:00 UTC,"""01-ai""","""yi-1.5-34b-chat""",4096.0,"""01-ai/Yi-1.5-34B-Chat""",null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""01-ai/yi-1.5-34b-chat""","""01-ai""","""Yi-1.5-34B-Chat"""
"""01-ai/yi-1.5-34b-chat""",2024-11-05 00:00:00 UTC,"""01-ai""","""yi-1.5-34b-chat""",4096.0,"""01-ai/Yi-1.5-34B-Chat""",null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""01-ai/yi-1.5-34b-chat""","""01-ai""","""Yi-1.5-34B-Chat"""
"""01-ai/yi-1.5-34b-chat""",2024-11-06 00:00:00 UTC,"""01-ai""","""yi-1.5-34b-chat""",4096.0,"""01-ai/Yi-1.5-34B-Chat""",null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""01-ai/yi-1.5-34b-chat""","""01-ai""","""Yi-1.5-34B-Chat"""
"""01-ai/yi-1.5-34b-chat""",2024-11-07 00:00:00 UTC,"""01-ai""","""yi-1.5-34b-chat""",4096.0,"""01-ai/Yi-1.5-34B-Chat""",null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""01-ai/yi-1.5-34b-chat""","""01-ai""","""Yi-1.5-34B-Chat"""
"""01-ai/yi-1.5-34b-chat""",2024-11-08 00:00:00 UTC,"""01-ai""","""yi-1.5-34b-chat""",4096.0,"""01-ai/Yi-1.5-34B-Chat""",null,null,null,null,null,null,null,null,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""01-ai/yi-1.5-34b-chat""","""01-ai""","""Yi-1.5-34B-Chat"""


In [23]:
import polars as pl

df = pl.read_parquet("data/openrouter_panel.parquet")

# ── Basic shape ──────────────────────────────────────────────────────────────
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(f"Unique models (openrouter_id): {df['openrouter_id'].n_unique():,}")
print(f"Unique authors:                {df['openrouter_author'].n_unique():,}")
print(f"Unique endpoint providers:     {df['endpoint_provider'].n_unique():,}")
print()

# ── Null rates for key columns ────────────────────────────────────────────────
key_cols = [
    "net_revenue", "volume", "count",
    "total_prompt_tokens", "total_completion_tokens",
    "prompt_price", "completion_price",
    "huggingface_id",
]
print("Null rates (key columns):")
print(
    df.select([pl.col(c).is_null().mean().alias(c) for c in key_cols])
    .unpivot(variable_name="column", value_name="null_rate")
    .sort("null_rate", descending=True)
)
print()

# ── Usage coverage ────────────────────────────────────────────────────────────
has_volume = df.filter(pl.col("volume").is_not_null())
print(f"Rows with volume data: {len(has_volume):,} ({len(has_volume)/len(df):.1%})")
print(f"Total volume (tokens): {has_volume['volume'].sum():,.0f}")
print(f"Total net_revenue ($): {has_volume['net_revenue'].sum():,.2f}")
print()

# ── Top 15 models by total volume ─────────────────────────────────────────────
# ── Top 15 models by total volume ─────────────────────────────────────────────
print("Top 15 models by total volume:")
print(
    has_volume
    .group_by("openrouter_id")
    .agg(
        pl.col("volume").sum().alias("total_volume"),
        pl.col("net_revenue").sum().alias("total_revenue"),
        pl.col("date").n_unique().alias("days_active"),
        pl.col("prompt_price").drop_nulls().last().alias("prompt_price"),
        pl.col("completion_price").drop_nulls().last().alias("completion_price"),
    )
    .sort("total_volume", descending=True)
    .head(15)
)
print()

# ── Top 10 authors by volume & revenue ───────────────────────────────────────
print("Top 10 authors by total volume:")
print(
    has_volume
    .group_by("openrouter_author")
    .agg(
        pl.col("volume").sum().alias("total_volume"),
        pl.col("net_revenue").sum().alias("total_revenue"),
        pl.col("openrouter_id").n_unique().alias("num_models"),
    )
    .sort("total_volume", descending=True)
    .head(10)
)


# ── Monthly aggregate ─────────────────────────────────────────────────────────
print("Monthly totals:")
print(
    has_volume
    .with_columns(pl.col("date").dt.truncate("1mo").alias("month"))
    .group_by("month")
    .agg(
        pl.col("volume").sum().alias("total_volume"),
        pl.col("net_revenue").sum().alias("total_revenue"),
        pl.col("openrouter_id").n_unique().alias("active_models"),
    )
    .sort("month")
)
print()

# ── Price distribution ────────────────────────────────────────────────────────
print("Price distribution ($/token, non-zero rows):")
print(
    df.filter(pl.col("prompt_price").is_not_null() & (pl.col("prompt_price") > 0))
    .select("prompt_price", "completion_price")
    .describe()
)
print()

# ── Top endpoint providers ────────────────────────────────────────────────────
print("Top 15 endpoint providers by row count:")
print(
    df.group_by("endpoint_provider")
    .agg(pl.len().alias("rows"))
    .sort("rows", descending=True)
    .head(15)
)


Shape: 311,422 rows × 48 cols
Date range: 2023-11-23 00:00:00+00:00 → 2026-03-25 00:00:00+00:00
Unique models (openrouter_id): 890
Unique authors:                119
Unique endpoint providers:     89

Null rates (key columns):
shape: (8, 2)
┌─────────────────────────┬───────────┐
│ column                  ┆ null_rate │
│ ---                     ┆ ---       │
│ str                     ┆ f64       │
╞═════════════════════════╪═══════════╡
│ net_revenue             ┆ 0.989933  │
│ volume                  ┆ 0.864528  │
│ count                   ┆ 0.662744  │
│ total_prompt_tokens     ┆ 0.662744  │
│ total_completion_tokens ┆ 0.662744  │
│ prompt_price            ┆ 0.53028   │
│ completion_price        ┆ 0.53028   │
│ huggingface_id          ┆ 0.403668  │
└─────────────────────────┴───────────┘

Rows with volume data: 42,189 (13.5%)
Total volume (tokens): 215,050,215
Total net_revenue ($): -9,128.57

Top 15 models by total volume:
shape: (15, 6)
┌───────────────────┬──────────────┬─────────

In [24]:
# ── Token distribution check ──────────────────────────────────────────────────
token_cols = ["volume", "total_prompt_tokens", "total_completion_tokens", "total_native_tokens_reasoning"]

print("Token column coverage:")
print(
    df.select([pl.col(c).is_not_null().sum().alias(c) for c in token_cols])
    .unpivot(variable_name="column", value_name="non_null_rows")
)
print()

print("Token distribution (non-null rows):")
print(df.select(token_cols).describe())
print()

# Correlation between volume and token components (rows where all are present)
complete = df.filter(pl.all_horizontal([pl.col(c).is_not_null() for c in token_cols]))
print(f"Rows with all four token cols: {len(complete):,}")
if len(complete) > 0:
    print()
    print("Sum check — do prompt+completion+reasoning ≈ volume?")
    print(
        complete
        .with_columns(
            (pl.col("total_prompt_tokens") + pl.col("total_completion_tokens") + pl.col("total_native_tokens_reasoning")).alias("sum_parts")
        )
        .select(
            pl.col("volume").sum().alias("total_volume"),
            pl.col("sum_parts").sum().alias("total_parts_sum"),
            (pl.col("sum_parts") / pl.col("volume")).mean().alias("ratio_mean"),
            (pl.col("sum_parts") / pl.col("volume")).median().alias("ratio_median"),
        )
    )


Token column coverage:
shape: (4, 2)
┌───────────────────────────────┬───────────────┐
│ column                        ┆ non_null_rows │
│ ---                           ┆ ---           │
│ str                           ┆ u32           │
╞═══════════════════════════════╪═══════════════╡
│ volume                        ┆ 42189         │
│ total_prompt_tokens           ┆ 105029        │
│ total_completion_tokens       ┆ 105029        │
│ total_native_tokens_reasoning ┆ 100767        │
└───────────────────────────────┴───────────────┘

Token distribution (non-null rows):
shape: (9, 5)
┌────────────┬───────────────┬─────────────────────┬───────────────────────┬───────────────────────┐
│ statistic  ┆ volume        ┆ total_prompt_tokens ┆ total_completion_toke ┆ total_native_tokens_r │
│ ---        ┆ ---           ┆ ---                 ┆ ns                    ┆ easoning              │
│ str        ┆ f64           ┆ f64                 ┆ ---                   ┆ ---                   │
│       

In [25]:
has_volume.group_by("openrouter_id").agg(
        pl.col("volume").sum().alias("total_volume"),
        pl.col("net_revenue").sum().alias("total_revenue"),
        pl.col("date").n_unique().alias("days_active"),
        pl.col("prompt_price").drop_nulls().last().alias("prompt_price"),
        pl.col("completion_price").drop_nulls().last().alias("completion_price"),
    ).sort("total_volume", descending=True).head(15)['openrouter_id'].to_list()

['anthropic/claude-3-7-sonnet-20250219',
 'anthropic/claude-4-sonnet-20250522',
 'google/gemini-2.5-pro',
 'google/gemini-2.5-pro-preview-03-25',
 'anthropic/claude-4-opus-20250522',
 'anthropic/claude-3.5-sonnet',
 'anthropic/claude-3-7-sonnet-20250219:thinking',
 'openai/gpt-4.1-2025-04-14',
 'deepseek/deepseek-chat-v3-0324',
 'anthropic/claude-3.5-sonnet:beta',
 'x-ai/grok-4-07-09',
 'google/gemini-2.5-flash',
 'anthropic/claude-3-7-sonnet-20250219:beta',
 'openai/gpt-4o-mini',
 'anthropic/claude-4.1-opus-20250805']

In [27]:
df.filter(pl.col('openrouter_id') == 'anthropic/claude-3-7-sonnet-20250219')

openrouter_id,date,openrouter_author,openrouter_model_name,context_length,huggingface_id,prompt_price,completion_price,image_price,request_price,web_search_price,internal_reasoning_price,image_output_price,discount_price,hidden,endpoint_is_hidden,endpoint_is_deranked,endpoint_is_disabled,endpoint_provider,input_cache_read_price,audio_price,input_audio_cache_price,input_cache_write_price,image_token_price,audio_output_price,line_items_price,companyDiscount_price,vendorDiscount_price,model_permaslug,variant,variant_permaslug,count,total_completion_tokens,total_prompt_tokens,total_native_tokens_reasoning,num_media_prompt,num_media_completion,num_audio_prompt,total_native_tokens_cached,total_tool_calls,requests_with_tool_call_errors,model,volume,net_revenue,total_tokens,openrouter_id_clean,huggingface_author,huggingface_model_name
str,"datetime[ms, UTC]",str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,str,str,str
"""anthropic/claude-3-7-sonnet-20…",2025-02-24 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219""",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2025-02-25 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219""",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2025-02-26 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219""",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,null,null,810490.0,4.17238485e8,2.9114e10,1.762443e6,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…",51001.069044,null,2.9531e10,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2025-02-27 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219""",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,null,null,1.805909e6,9.32986573e8,6.5302e10,7.863742e6,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…",100234.058522,null,6.6235e10,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2025-02-28 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219""",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet""",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""anthropic/claude-3-7-sonnet-20…",2026-03-21 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219""",200000.0,null,0.000003,0.000015,null,null,0.01,null,null,0.0,0.0,0.0,0.0,0.0,"""Google""",0.0000003,null,null,0.000004,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…",null,"""anthropic/claude-3-7-sonnet-20…",1.277558e6,6.79789838e8,1.3604e10,1.2205954e7,166535.0,0.0,0.0,5.3459e9,179748.0,7429.0,null,null,null,1.4284e10,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2026-03-22 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219""",200000.0,null,0.000003,0.000015,null,null,0.01,null,null,0.0,0.0,0.0,0.0,0.0,"""Google""",0.0000003,null,null,0.000004,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…",null,"""anthropic/claude-3-7-sonnet-20…",1.245123e6,6.

In [ ]:
df.filter(pl.col('openrouter_id') == 'anthropic/claude-3-7-sonnet-20250219:thinking')

openrouter_id,date,openrouter_author,openrouter_model_name,context_length,huggingface_id,prompt_price,completion_price,image_price,request_price,web_search_price,internal_reasoning_price,image_output_price,discount_price,hidden,endpoint_is_hidden,endpoint_is_deranked,endpoint_is_disabled,endpoint_provider,input_cache_read_price,audio_price,input_audio_cache_price,input_cache_write_price,image_token_price,audio_output_price,line_items_price,companyDiscount_price,vendorDiscount_price,model_permaslug,variant,variant_permaslug,count,total_completion_tokens,total_prompt_tokens,total_native_tokens_reasoning,num_media_prompt,num_media_completion,num_audio_prompt,total_native_tokens_cached,total_tool_calls,requests_with_tool_call_errors,model,volume,net_revenue,total_tokens,openrouter_id_clean,huggingface_author,huggingface_model_name
str,"datetime[ms, UTC]",str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,str,str,str
"""anthropic/claude-3-7-sonnet-20…",2025-02-26 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219:thi…",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,"""thinking""",null,18158.0,1.8056393e7,5.91138951e8,8.53143e6,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…",2134.312457,null,6.09195344e8,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2025-02-27 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219:thi…",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,"""thinking""",null,151987.0,1.57202621e8,5.5375e9,7.4971673e7,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…",14807.990609,null,5.6947e9,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2025-02-28 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219:thi…",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,"""thinking""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2025-03-01 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219:thi…",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,"""thinking""",null,380831.0,4.1726831e8,1.4166e10,1.90565659e8,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…",35069.57762,null,1.4583e10,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2025-03-02 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219:thi…",200000.0,null,0.000003,0.000015,0.0048,0.0,null,null,null,null,0.0,0.0,0.0,0.0,"""Anthropic""",null,null,null,null,null,null,null,null,null,null,"""thinking""",null,525403.0,5.94911545e8,1.9822e10,2.57561381e8,null,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…",46205.81581,null,2.0416e10,"""anthropic/claude-3-7-sonnet""",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""anthropic/claude-3-7-sonnet-20…",2026-03-21 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219:thi…",200000.0,null,0.000003,0.000015,null,null,0.01,null,null,null,null,null,null,null,null,0.0000003,null,null,0.000004,null,null,null,null,null,"""anthropic/claude-3-7-sonnet-20…","""thinking""","""anthropic/claude-3-7-sonnet-20…",89078.0,9.4750962e7,1.7444e9,4.5188512e7,101015.0,0.0,0.0,7.22556463e8,24758.0,1041.0,null,null,null,1.8392e9,"""anthropic/claude-3-7-sonnet""",null,null
"""anthropic/claude-3-7-sonnet-20…",2026-03-22 00:00:00 UTC,"""anthropic""","""claude-3-7-sonnet-20250219:thi…",200000.0,null,0.000003,0.0